In [5]:
%%bash
pip install transformers datasets torchaudio soundfile

In [6]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch
import soundfile as sf

In [13]:
model_name = "wav2vec2-xls-r-300m"

processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)

audio, sr = sf.read("sample.wav")

if len(audio.shape) > 1:
    audio = audio.mean(axis=1)

if sr != 16000:
    import torchaudio
    resampler = torchaudio.transforms.Resample(sr, 16000)
    audio = resampler(torch.tensor(audio, dtype=torch.float32)).numpy()

input_values = processor(audio, sampling_rate=16000, return_tensors="pt").input_values

with torch.no_grad():
    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(predicted_ids)

print(transcription)

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

['usper in char kavyunka gehra as arhe']


In [27]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicVoices",
    "hindi",
    split="train",
    streaming=True
)

samples = list(dataset.take(3))

for s in samples:
    print(s["text"])

Resolving data files:   0%|          | 0/88 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

घर
तीन
आज का समय बहुत ही डिजिटल समय हो गया है पहले क्या था कि पहले मोबाइल सबके हाथ में नहीं था और नया नया मोबाइल नहीं आया था छोटा मोबाइल था


In [28]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicVoices",
    "hindi",
    split="train",
    streaming=True
)

# Collect some text samples
texts = []
for i, sample in enumerate(dataset):
    texts.append(sample["text"])
    if i >= 1000:   # enough for tokenizer
        break

print(texts[:5])

Resolving data files:   0%|          | 0/88 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

['घर', 'तीन', 'आज का समय बहुत ही डिजिटल समय हो गया है पहले क्या था कि पहले मोबाइल सबके हाथ में नहीं था और नया नया मोबाइल नहीं आया था छोटा मोबाइल था', 'इससे क्या होता था था अड़ोसी पड़ोसी किसी का भी फोन आता था तो पड़ोसी बात कराते थे व्यवहार अच्छे रहते थे', 'की हमारे किसी का फोन आया तो बात करा दिए और आज क्या हो गया है कि हाथों हाथ सभी के हाथों में जो है बड़े मोबाइल आ गए हैं']


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicVoices",
    "hindi",
    split="train",
    streaming=True
)

# Collect some text samples
texts = []
for i, sample in enumerate(dataset):
    texts.append(sample["text"])
    if i >= 1000:   # enough for tokenizer
        break

print(texts[:5])

In [34]:
vocab = set()

for text in texts:
    for char in text:
        # Keep only Devanagari + space
        if (
            char == " " or
            "\u0900" <= char <= "\u097F"   # Unicode range for Hindi
        ):
            vocab.add(char)

vocab = sorted(list(vocab))

print(vocab)
print("Total chars:", len(vocab))

[' ', 'ँ', 'ं', 'ः', 'अ', 'आ', 'इ', 'ई', 'उ', 'ऊ', 'ऋ', 'ऍ', 'ए', 'ऐ', 'ऑ', 'ओ', 'औ', 'क', 'ख', 'ग', 'घ', 'च', 'छ', 'ज', 'झ', 'ञ', 'ट', 'ठ', 'ड', 'ढ', 'ण', 'त', 'थ', 'द', 'ध', 'न', 'प', 'फ', 'ब', 'भ', 'म', 'य', 'र', 'ल', 'व', 'श', 'ष', 'स', 'ह', '़', 'ा', 'ि', 'ी', 'ु', 'ू', 'ृ', 'े', 'ै', 'ॉ', 'ो', 'ौ', '्']
Total chars: 62


In [35]:
vocab_dict = {char: i for i, char in enumerate(vocab)}

# Add special tokens
vocab_dict["[PAD]"] = len(vocab_dict)
vocab_dict["[UNK]"] = len(vocab_dict)

print(vocab_dict)

{' ': 0, 'ँ': 1, 'ं': 2, 'ः': 3, 'अ': 4, 'आ': 5, 'इ': 6, 'ई': 7, 'उ': 8, 'ऊ': 9, 'ऋ': 10, 'ऍ': 11, 'ए': 12, 'ऐ': 13, 'ऑ': 14, 'ओ': 15, 'औ': 16, 'क': 17, 'ख': 18, 'ग': 19, 'घ': 20, 'च': 21, 'छ': 22, 'ज': 23, 'झ': 24, 'ञ': 25, 'ट': 26, 'ठ': 27, 'ड': 28, 'ढ': 29, 'ण': 30, 'त': 31, 'थ': 32, 'द': 33, 'ध': 34, 'न': 35, 'प': 36, 'फ': 37, 'ब': 38, 'भ': 39, 'म': 40, 'य': 41, 'र': 42, 'ल': 43, 'व': 44, 'श': 45, 'ष': 46, 'स': 47, 'ह': 48, '़': 49, 'ा': 50, 'ि': 51, 'ी': 52, 'ु': 53, 'ू': 54, 'ृ': 55, 'े': 56, 'ै': 57, 'ॉ': 58, 'ो': 59, 'ौ': 60, '्': 61, '[PAD]': 62, '[UNK]': 63}


In [36]:
import json

with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

In [37]:
from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token=" "
)

In [38]:
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

In [39]:
from transformers import Wav2Vec2Processor

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

In [40]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    vocab_size=len(tokenizer)
)

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [41]:
print(model.lm_head.out_features)
print(len(tokenizer))

66
66


In [44]:
sample = next(iter(dataset))
print(sample.keys())
print(sample)

dict_keys(['audio_filepath', 'text', 'duration', 'lang', 'samples', 'verbatim', 'normalized', 'speaker_id', 'scenario', 'task_name', 'gender', 'age_group', 'job_type', 'qualification', 'area', 'district', 'state', 'occupation', 'verification_report', 'unsanitized_verbatim', 'unsanitized_normalized'])
{'audio_filepath': <datasets.features._torchcodec.AudioDecoder object at 0x7e1a4e3f8470>, 'text': 'घर', 'duration': 0.404, 'lang': 'hi', 'samples': 6464, 'verbatim': 'घर', 'normalized': 'घर', 'speaker_id': 'S4259147600351938', 'scenario': 'Read', 'task_name': 'Keywords Spotting', 'gender': 'Male', 'age_group': '18-30', 'job_type': 'Student', 'qualification': 'Undergrad and Grad.', 'area': 'Urban', 'district': 'Pratapgarh', 'state': 'Uttar Pradesh', 'occupation': 'Student', 'verification_report': "{'sst': False, 'comments': '', 'decision': 'excellent', 'book_read': False, 'off_topic': False, 'low_volume': False, 'stretching': False, 'long_pauses': False, 'echo_present': False, 'wrong_gender

In [45]:
def prepare_batch(batch):
    audio = batch["audio_filepath"]

    # Extract array + sampling rate
    audio_array = audio["array"]
    sampling_rate = audio["sampling_rate"]

    # Convert stereo → mono if needed
    if len(audio_array.shape) > 1:
        audio_array = audio_array.mean(axis=1)

    # Resample if needed
    if sampling_rate != 16000:
        import torch, torchaudio
        audio_array = torch.tensor(audio_array, dtype=torch.float32)
        resampler = torchaudio.transforms.Resample(sampling_rate, 16000)
        audio_array = resampler(audio_array).numpy()

    # Process audio
    batch["input_values"] = processor(
        audio_array,
        sampling_rate=16000
    ).input_values[0]

    # Process text
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

In [46]:
# processed_samples = []

# for i, sample in enumerate(dataset):
#     processed_samples.append(prepare_batch(sample))

#     if i >= 10:
#         break

def preprocess_dataset(example):
    return prepare_batch(example)

train_dataset = dataset.take(50)  # small subset
train_dataset = list(train_dataset)

train_dataset = [preprocess_dataset(x) for x in train_dataset]

In [47]:
print(processed_samples[0].keys())
print(processed_samples[0]["labels"])

dict_keys(['audio_filepath', 'text', 'duration', 'lang', 'samples', 'verbatim', 'normalized', 'speaker_id', 'scenario', 'task_name', 'gender', 'age_group', 'job_type', 'qualification', 'area', 'district', 'state', 'occupation', 'verification_report', 'unsanitized_verbatim', 'unsanitized_normalized', 'input_values', 'labels'])
[20, 42]


In [48]:
train_data = processed_samples[:10]

In [59]:
from dataclasses import dataclass
from typing import List, Dict, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_values = [f["input_values"] for f in features]
        labels = [f["labels"] for f in features]

        # Pad audio inputs
        batch = self.processor.feature_extractor.pad(
            {"input_values": input_values},
            return_tensors="pt"
        )

        # Pad labels directly using tokenizer
        labels_batch = self.processor.tokenizer.pad(
            {"input_ids": labels},
            return_tensors="pt",
            padding=True
        )

        # Replace padding with -100 (important for CTC loss)
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        batch["labels"] = labels
        return batch

In [60]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./wav2vec2-indic",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    learning_rate=1e-4,
    logging_steps=1,
    save_steps=10,
    fp16=True
)

In [61]:
data_collator = DataCollatorCTCWithPadding(processor=processor)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    data_collator=data_collator,
)

In [1]:
trainer.train()

NameError: name 'trainer' is not defined